In [ ]:
!pip install torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q -U transformers==4.44.2 peft==0.12.0 accelerate==0.34.2 bitsandbytes==0.43.3 trl==0.10.1 datasets

In [ ]:
!git clone https://github.com/DimitrisKu/Active-Reading--Pattern-Recognition.git

import os

%cd /content/Active-Reading--Pattern-Recognition

print("Current Directory:", os.getcwd())

In [ ]:
# Connect to hugging face (Add a token with name "HF_TOKEN" from Hugging Face into Secrets here in Colab)
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Successfully logged into Hugging Face.")
except userdata.SecretNotFoundError:
    print("HF_TOKEN not found in Colab secrets. Please add it to access gated models.")
except Exception as e:
    print(f"An error occurred during Hugging Face login: {e}")

In [ ]:
!pip install --upgrade transformers tokenizers accelerate

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
)

# LoRA
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)
model.gradient_checkpointing_enable()

model.print_trainable_parameters()

FINE-TUNING FOR **PARAPHRASE** USING GENERATED CORPUS

---

In [2]:
import json
import glob
from datasets import Dataset

ORIGINAL_CORPUS = "Datasets/simple_wiki_corpus.json"
GENERATED_PARAPHRASE = "Datasets/generated_simplewiki/paraphrase_outputs/*.jsonl"

print("Loading original corpus...")
with open(ORIGINAL_CORPUS, "r", encoding="utf-8") as f:
    original_list = json.load(f)
    original_dict = {item['doc_name']: item['text'] for item in original_list}

data_for_tuning = []

for file_path in file_paths:
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try:
                entry = json.loads(line)
                paraphrased_text = entry.get("text", "").strip()

                if paraphrased_text:
                    data_for_tuning.append({
                        "text": paraphrased_text
                    })
            except json.JSONDecodeError:
                continue

dataset_paraphrase = Dataset.from_list(data_for_tuning)
print(f"Total samples for training: {len(dataset_paraphrase)}")

FileNotFoundError: [Errno 2] No such file or directory: 'Datasets/generated_simplewiki/paraphrase_outputs/*.jsonl'

In [ ]:
from datasets import load_dataset, Dataset, concatenate_datasets

def format_active_reading(example):
    return {"text": example['output']}

formatted_dataset_paraphrase = dataset_paraphrase.map(
    format_active_reading,
    remove_columns=dataset_paraphrase.column_names
)

# DCLM 10% mixing
dataset_dclm = load_dataset("mlfoundations/dclm-baseline-1.0", split="train", streaming=True)

num_paraphrase = len(formatted_dataset_paraphrase)
num_dclm_needed = max(1, num_paraphrase // 9) # 10%

print(f"Paraphrase samples: {num_paraphrase}")
print(f"DCLM samples needed: {num_dclm_needed}")

dclm_samples = []
for i, example in enumerate(dataset_dclm.take(num_dclm_needed)):
    dclm_samples.append({"text": example['text']})

dataset_dclm_final = Dataset.from_list(dclm_samples)

# Mixing & Shuffling
final_dataset = concatenate_datasets([formatted_dataset_paraphrase, dataset_dclm_final])
final_dataset = final_dataset.shuffle(seed=42)

print(f"Final combined dataset size: {len(final_dataset)}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

sft_config = SFTConfig(
    output_dir="/content/drive/MyDrive/qwen_paraphrase_checkpoints",
    max_seq_length=1024,
    packing=True,
    dataset_text_field="text",
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=3e-4,
    warmup_ratio=0.1,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    optim="paged_adamw_32bit",
    logging_steps=5,
    fp16=True,
    seed=42,
    gradient_checkpointing=True,
    report_to="none"
)

model.enable_input_require_grads()
trainer = SFTTrainer(
    model=model,
    train_dataset=final_dataset,
    args=sft_config,
)

trainer.train(resume_from_checkpoint=False) # Change this to True after first run

EVALUATION OF THE MODEL FOR THE PARAPHRASE TASK

---